<a href="https://colab.research.google.com/github/michalis0/MGT-502-Data-Science-and-Machine-Learning/blob/main/07_Gen-AI/2-chatgpt-prompt-engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Getting Started with Prompt Engineering

(credits DAIR.AI)


This notebook contains examples and exercises to learning about prompt engineering.

We will be using the [OpenAI APIs](https://platform.openai.com/) for all examples. I am using the default settings `temperature=0.7` and `top-p=1`

---

## 1. Prompt Engineering Basics

Objectives
- Load the libraries
- Review the format
- Cover basic prompts
- Review common use cases

Below we are loading the necessary libraries, utilities, and configurations.

In [85]:

# update or install the necessary libraries
!pip install -q --upgrade openai
!pip install -q --upgrade langchain
!pip install -q --upgrade langchain_openai
!pip install -q --upgrade langchain_community
!pip install -q --upgrade langchain_text_splitters
!pip install -q --upgrade python_dotenv
!pip install -q chromadb

In [74]:
from openai import OpenAI
import os
import IPython
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAI as LangChainOpenAI

Load environment variables. You can use anything you like but I used `python-dotenv`. Just create a `.env` file with your `OPENAI_API_KEY` then load it. 

- Go to https://serper.dev/ to get the API key to do google searches if you want to execute some of the cells down below.

In [95]:
%env OPENAI_API_KEY=

env: OPENAI_API_KEY=


In [96]:
%env SERPAPI_API_KEY=

env: SERPAPI_API_KEY=


In [8]:
load_dotenv()

# API configuration
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# for LangChain
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")
os.environ["SERPAPI_API_KEY"] = os.getenv("SERPAPI_API_KEY", "")

In [15]:
def set_open_params(
    model="gpt-5.4-mini",
    temperature=0.7,
    max_completion_tokens=256,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0,
):
    """Set OpenAI parameters."""

    openai_params = {}    

    openai_params['model'] = model
    openai_params['temperature'] = temperature
    openai_params['max_completion_tokens'] = max_completion_tokens
    openai_params['top_p'] = top_p
    openai_params['frequency_penalty'] = frequency_penalty
    openai_params['presence_penalty'] = presence_penalty
    return openai_params

def get_completion(params, prompt):
    """Get a chat completion from the OpenAI API."""

    response = client.chat.completions.create(
        model=params['model'],
        messages=[
            {"role": "user", "content": prompt},
        ],
        temperature=params['temperature'],
        max_completion_tokens=params['max_completion_tokens'],
        top_p=params['top_p'],
        frequency_penalty=params['frequency_penalty'],
        presence_penalty=params['presence_penalty'],
    )
    return response.choices[0].message.content

Basic prompt example:

In [16]:
# basic example
params = set_open_params()

prompt = "The sky is"

response = get_completion(params, prompt)

In [ ]:
response

" blue\n\nThe sky is blue because of the way the Earth's atmosphere scatters sunlight. The blue color is caused by the shorter wavelengths of blue light being scattered more effectively by the atmosphere than the longer wavelengths of other colors."

In [17]:
IPython.display.Markdown(response)

blue in clear daylight.

Try with different temperature to compare results:

In [18]:
params = set_open_params(temperature=0)
prompt = "The sky is"
response = get_completion(params, prompt)
IPython.display.Markdown(response)

The sky is usually blue during the day.

### 1.1 Text Summarization

In [19]:
params = set_open_params(temperature=0.7)
prompt = """Antibiotics are a type of medication used to treat bacterial infections. They work by either killing the bacteria or preventing them from reproducing, allowing the body's immune system to fight off the infection. Antibiotics are usually taken orally in the form of pills, capsules, or liquid solutions, or sometimes administered intravenously. They are not effective against viral infections, and using them inappropriately can lead to antibiotic resistance. 

Explain the above in one sentence:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

Antibiotics are medicines used to treat bacterial infections by killing bacteria or stopping them from multiplying, but they do not work against viruses and should be used carefully to avoid resistance.

Exercise: Instruct the model to explain the paragraph in one sentence like "I am 5". Do you see any differences?

### 1.2 Question Answering

In [20]:
prompt = """Answer the question based on the context below. Keep the answer short and concise. Respond "Unsure about answer" if not sure about the answer.

Context: Teplizumab traces its roots to a New Jersey drug company called Ortho Pharmaceutical. There, scientists generated an early version of the antibody, dubbed OKT3. Originally sourced from mice, the molecule was able to bind to the surface of T cells and limit their cell-killing potential. In 1986, it was approved to help prevent organ rejection after kidney transplants, making it the first therapeutic antibody allowed for human use.

Question: What was OKT3 originally sourced from?

Answer:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)


Mice

Context obtained from here: https://www.nature.com/articles/d41586-023-00400-x

Exercise: Edit prompt and get the model to respond that it isn't sure about the answer. 

### 1.3 Text Classification

In [21]:
prompt = """Classify the text into neutral, negative or positive.

Text: I think the food was okay.

Sentiment:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

neutral

Exercise: Modify the prompt to instruct the model to provide an explanation to the answer selected. 

### 1.4 Role Playing

In [57]:
params = set_open_params(max_completion_tokens=1000)

prompt = """The following is a conversation with an AI research assistant. The assistant tone is technical and scientific. Try to c

Human: Hello, who are you?
AI: Greeting! I am an AI research assistant. How can I help you today?
Human: Can you tell me about the creation of blackholes?
AI:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

Black holes form when a sufficient amount of mass is compressed into a very small region, causing gravity to become so strong that nothing—not even light—can escape.

The most common formation pathway is **stellar collapse**:

1. **Massive star burns its nuclear fuel**  
   A star maintains balance between outward pressure from fusion and inward gravitational pull.

2. **Fuel runs out**  
   Once the star can no longer fuse elements to produce enough pressure, gravity dominates.

3. **Core collapses**  
   If the remaining core is massive enough—typically above the limit for a neutron star—the collapse continues.

4. **Event horizon forms**  
   The object becomes dense enough that an event horizon appears, creating a black hole.

Other formation channels include:

- **Merging neutron stars or black holes**
- **Direct collapse of very massive gas clouds**
- **Possible primordial black holes**, which are hypothetical and may have formed in the early universe

If you want, I can also explain:
- the difference between stellar and supermassive black holes,
- what happens near the event horizon,
- or how black holes are detected indirectly.

Exercise: Modify the prompt to instruct the model to keep AI responses concise and short.

### 1.5 Code Generation

In [24]:
prompt = "\"\"\"\nTable departments, columns = [DepartmentId, DepartmentName]\nTable students, columns = [DepartmentId, StudentId, StudentName]\nCreate a MySQL query for all students in the Computer Science Department\n\"\"\""

response = get_completion(params, prompt)
IPython.display.Markdown(response)


```sql
SELECT s.StudentId, s.StudentName
FROM students s
JOIN departments d
  ON s.DepartmentId = d.DepartmentId
WHERE d.DepartmentName = 'Computer Science';
```

If you want, I can also provide a version that returns all columns from both tables.

### 1.6 Reasoning

In [25]:
prompt = """The odd numbers in this group add up to an even number: 15, 32, 5, 13, 82, 7, 1. 

Solve by breaking the problem into steps. First, identify the odd numbers, add them, and indicate whether the result is odd or even."""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

Step 1: Identify the odd numbers in the group.

Numbers: 15, 32, 5, 13, 82, 7, 1  
Odd numbers: **15, 5, 13, 7, 1**

Step 2: Add the odd numbers.

15 + 5 = 20  
20 + 13 = 33  
33 + 7 = 40  
40 + 1 = **41**

Step 3: Determine whether the result is odd or even.

**41 is odd.**

So, the odd numbers add up to **41**, which is **odd**, not even.

Exercise: Improve the prompt to have a better structure and output format.

## 2. Advanced Prompting Techniques

Objectives:

- Cover more advanced techniques for prompting: few-shot, chain-of-thoughts,...

### 2.2 Few-shot prompts

In [26]:
prompt = """The odd numbers in this group add up to an even number: 4, 8, 9, 15, 12, 2, 1.
A: The answer is False.

The odd numbers in this group add up to an even number: 17,  10, 19, 4, 8, 12, 24.
A: The answer is True.

The odd numbers in this group add up to an even number: 16,  11, 14, 4, 8, 13, 24.
A: The answer is True.

The odd numbers in this group add up to an even number: 17,  9, 10, 12, 13, 4, 2.
A: The answer is False.

The odd numbers in this group add up to an even number: 15, 32, 5, 13, 82, 7, 1. 
A:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

The answer is **True**.

Odd numbers: **15, 5, 13, 7, 1**  
Sum: **15 + 5 + 13 + 7 + 1 = 41**, which is **odd**.

So the statement “the odd numbers in this group add up to an even number” is **False**.

### 2.3 Chain-of-Thought (CoT) Prompting

In [27]:
prompt = """The odd numbers in this group add up to an even number: 4, 8, 9, 15, 12, 2, 1.
A: Adding all the odd numbers (9, 15, 1) gives 25. The answer is False.

The odd numbers in this group add up to an even number: 15, 32, 5, 13, 82, 7, 1. 
A:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

Adding all the odd numbers (15, 5, 13, 7, 1) gives 41. The answer is False.

### 2.4 Zero-shot CoT

In [28]:
prompt = """I went to the market and bought 10 apples. I gave 2 apples to the neighbor and 2 to the repairman. I then went and bought 5 more apples and ate 1. How many apples did I remain with?

Let's think step by step."""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

Let’s go step by step:

1. You bought **10 apples**.
2. You gave away **2** to the neighbor and **2** to the repairman:
   - 10 - 2 - 2 = **6**
3. You bought **5 more apples**:
   - 6 + 5 = **11**
4. You ate **1 apple**:
   - 11 - 1 = **10**

**Answer: 10 apples**

### 2.6 PAL - Code as Reasoning

We are developing a simple application that's able to reason about the question being asked through code. 

Specifically, the application takes in some data and answers a question about the data input. The prompt includes a few exemplars which are adopted from [here](https://github.com/reasoning-machines/pal/blob/main/pal/prompt/penguin_prompt.py).  

In [47]:
# lm instance
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [48]:
question = "Which is the oldest penguin?"

In [51]:
PENGUIN_PROMPT = '''
# Instructions: You are a Python code generator. 
# Your task is to solve the penguin logic puzzle by writing executable Python code.
# Output ONLY raw Python code. Do NOT include markdown code blocks (```python), 
# Do NOT include explanations. The code must define a variable 'answer'.

"""
Q: Here is a table where the first line is a header and each subsequent line is a penguin:
name, age, height (cm), weight (kg) 
Louis, 7, 50, 11
Bernard, 5, 80, 13
Vincent, 9, 60, 11
Gwen, 8, 70, 15
For example: the age of Louis is 7, the weight of Gwen is 15 kg, the height of Bernard is 80 cm. 
We now add a penguin to the table:
James, 12, 90, 12
How many penguins are less than 8 years old?
"""
penguins = [('Louis', 7, 50, 11), ('Bernard', 5, 80, 13), ('Vincent', 9, 60, 11), ('Gwen', 8, 70, 15)]
penguins.append(('James', 12, 90, 12))
num_penguin_under_8 = len([p for p in penguins if p[1] < 8])
answer = num_penguin_under_8

"""
Q: Here is a table where the first line is a header and each subsequent line is a penguin:
name, age, height (cm), weight (kg) 
Louis, 7, 50, 11
Bernard, 5, 80, 13
Vincent, 9, 60, 11
Gwen, 8, 70, 15
For example: the age of Louis is 7, the weight of Gwen is 15 kg, the height of Bernard is 80 cm.
Which is the youngest penguin?
"""
penguins = [('Louis', 7, 50, 11), ('Bernard', 5, 80, 13), ('Vincent', 9, 60, 11), ('Gwen', 8, 70, 15)]
youngest_penguin_name = sorted(penguins, key=lambda x: x[1])[0][0]
answer = youngest_penguin_name

"""
{question}
"""
'''.strip() + '\n'

Now that we have the prompt and question. We can send it to the model. It should output the steps, in code, needed to get the solution to the answer.

In [52]:
llm_out = llm.invoke(PENGUIN_PROMPT.format(question=question)).content
print(llm_out)

penguins = [('Louis', 7, 50, 11), ('Bernard', 5, 80, 13), ('Vincent', 9, 60, 11), ('Gwen', 8, 70, 15)]
oldest_penguin_name = sorted(penguins, key=lambda x: x[1], reverse=True)[0][0]
answer = oldest_penguin_name


In [53]:
exec(llm_out)
print(answer)

Vincent


That's the correct answer! Vincent is the oldest penguin. 

Exercise: Try a different question and see what's the result.

---

# 3. Tools and Applications

Objective:

- Demonstrate how to use LangChain to demonstrate simple applications using prompting techniques and LLMs

### 3.2 Data-Augmented Generation

First, we need to download the data we want to use as source to augment generation.

Code example adopted from [LangChain Documentation](https://langchain.readthedocs.io/en/latest/modules/chains/combine_docs_examples/qa_with_sources.html). We are only using the examples for educational purposes.

Prepare the data first:

In [75]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings import CohereEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate

In [76]:
with open('./state_of_the_union.txt') as f:
    state_of_the_union = f.read()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_text(state_of_the_union)

embeddings = OpenAIEmbeddings()

In [66]:
import chromadb

In [67]:
docsearch = Chroma.from_texts(texts, embeddings, metadatas=[{"source": str(i)} for i in range(len(texts))])

In [68]:
query = "What did the president say about Justice Breyer"
docs = docsearch.similarity_search(query)

Let's quickly test it:

In [92]:
from langchain_classic.chains.qa_with_sources import load_qa_with_sources_chain
from langchain_openai import ChatOpenAI

In [93]:
chain = load_qa_with_sources_chain(ChatOpenAI(model="gpt-4o-mini", temperature=0), chain_type="stuff")
query = "What did the president say about Justice Breyer"
chain({"input_documents": docs, "question": query}, return_only_outputs=True)

{'output_text': "The president honored Justice Stephen Breyer, acknowledging him as a dedicated public servant, an Army veteran, and a Constitutional scholar. He expressed gratitude for Justice Breyer's service and mentioned that he had nominated Judge Ketanji Brown Jackson to succeed him, highlighting her qualifications and the support she has received.\n\nSOURCES: 31"}

Let's try a question with a custom prompt:

In [94]:
template = """Given the following extracted parts of a long document and a question, create a final answer with references ("SOURCES"). 
If you don't know the answer, just say that you don't know. Don't try to make up an answer.
ALWAYS return a "SOURCES" part in your answer.
Respond in Spanish.

QUESTION: {question}
=========
{summaries}
=========
FINAL ANSWER IN SPANISH:"""

# create a prompt template
PROMPT = PromptTemplate(template=template, input_variables=["summaries", "question"])

# query 
chain = load_qa_with_sources_chain(ChatOpenAI(model="gpt-4o-mini", temperature=0), chain_type="stuff", prompt=PROMPT)
query = "What did the president say about Justice Breyer?"
chain({"input_documents": docs, "question": query}, return_only_outputs=True)

{'output_text': 'El presidente honró al juez Stephen Breyer, describiéndolo como un veterano del ejército, un erudito constitucional y un juez que se retira de la Corte Suprema de los Estados Unidos. Agradeció a Breyer por su servicio y destacó su dedicación a servir al país. Además, mencionó que había nominado a la jueza Ketanji Brown Jackson para continuar el legado de excelencia de Breyer en la Corte Suprema.\n\nSOURCES: 31'}

Exercise: Try using a different dataset from the internet and try different prompt, including all the techniques you learned in the lecture.